# Dvouvrstvý perceptron pro obrázků CIFAR10

- Úkolem cvičení je naprogramovat **dvouvrstvý perceptron** pro klasifikaci obrázků na datasetu CIFAR-10.
- Jediný rozdíl oproti minulému cvičení, kde jsme trénovali lineární model, je tedy použitý klasifikátor.

In [ ]:
%load_ext autoreload
%autoreload 2

In [29]:
import sys
sys.path.append('..')  # import tests, ans

import numpy as np
import matplotlib.pyplot as plt
import PIL
import torch
import torchvision

import ans
from tests import test_two_layer_perceptron

# Data

## Preprocessing

- Preprocessing dat provedeme shodně s úlohou [`linear_classification`](linear_classification.ipynb).

### TODO: implementuje funkci `preprocess`.

In [30]:
def preprocess(img: PIL.Image) -> torch.Tensor:
    """
    Args:
        img: image of type PIL.Image
    Returns:
        x: 1-dimensional tensor; shape (num_pixels * num_channels,), dtype float32
    """
    
    ########################################
    
    one_dim_array = np.array(img).flatten()
    x = torch.tensor(one_dim_array, dtype=torch.float32) / 255
    
    ########################################
    
    return x

In [ ]:
test_two_layer_perceptron.TestPreprocess.eval(preprocess_fn=preprocess)

# Dvouvrstvý perceptron

- Model bude optimalizovat křížovou entropii
  $$
  l_n = -\log\frac{\exp{s_{n,y_n}}}{\sum_{k=1}^{K}{\exp{s_{n,k}}}}
  $$
  kde skóre $\boldsymbol{s}_n$ bude výsledek dopředného průchodu dvouvrstvého perceptronu se sigmoidou $\sigma(\cdot)$ jako aktivační funkcí
  $$
  \begin{align*}
  \boldsymbol{r}_n &= \boldsymbol{x}_n \cdot \boldsymbol{w}^{(1)} + \boldsymbol{b}^{(1)} \\
  \boldsymbol{h}_n &= \sigma(\boldsymbol{r}_n) \\
  \boldsymbol{s}_n &= \boldsymbol{x}_n \cdot \boldsymbol{w}^{(2)} + \boldsymbol{b}^{(2)} \\
  \end{align*}
  $$
- Klasifikátor bude implementován jako třída [`ans.classification.TwoLayerPerceptron`](../ans/classification.py).
- Třída obsahuje parametry klasifikátoru jako atributy
  | atribut   | značení                                                            | rozměr       |
  | --------- | ------------------------------------------------------------------ | ------------ |
  | `weight1` | $\boldsymbol{w}^{(1)} = \left[w_{d,k}^{(1)}\right]$                | $D \times H$ |
  | `bias1`   | $\boldsymbol{b}^{(1)} = \left[b_1^{(1)}, \ldots, b_H^{(1)}\right]$ | $H$          |
  | `weight2` | $\boldsymbol{w}^{(2)} = \left[w_{d,k}^{(2)}\right]$                | $H \times K$ |
  | `bias2`   | $\boldsymbol{b}^{(2)} = \left[b_1^{(2)}, \ldots, b_K^{(2)}\right]$ | $K$          |
- Třída bude mít shodné tři metody `__init__`, `train_step` a `val_step` jako `ans.classification.LinearSoftmaxModel`.
- Rozdílem bude pouze jejich vnitřní implementace.

### TODO: implementuje metodu [`ans.classification.TwoLayerPerceptron.__init__`](../ans/classification.py).

In [ ]:
test_two_layer_perceptron.TestInit.eval()

### TODO: implementuje funkci [`ans.classification.TwoLayerPerceptron.train_step`](../ans/classification.py).

In [ ]:
test_two_layer_perceptron.TestTrainStep.eval()

### TODO: implementuje funkci [`ans.classification.TwoLayerPerceptron.val_step`](../ans/classification.py).

In [ ]:
test_two_layer_perceptron.TestValStep.eval()

# Trénování klasifikátoru

- Nejlepší model uložte jako
  ``` python
  model.save('../output/two_layer_perceptron_weights.pt')
  ```

### TODO: Natrénujte dvouvrstvý perceptron tak, aby dosáhl alespoň 45 % (bonusově 50%) *validační* accuracy.

In [ ]:
ans.utils.seed_everything(0)

########################################
# hyperparametry
batch_size = 10
learning_rate = 1e-3
num_epochs = 200

# dataset
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=preprocess)
val_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=preprocess)

# loadery
train_loader = ans.data.BatchLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = ans.data.BatchLoader(val_dataset, batch_size=batch_size, shuffle=False)

# model
in_size = 32 * 32 * 3  # CIFAR-10 image size (3 color channels, 32x32 pixels)
hidden_size = 256  # neurons
out_size = 10  # CIFAR-10 has 10 classes
model = ans.classification.TwoLayerPerceptron(in_size=in_size, hidden_size=hidden_size, out_size=out_size)

# validace pred trenovanim (sanity check, loss by mel byt cca 2.30)
train_loss, train_acc = ans.classification.validate(model, train_loader)
val_loss, val_acc = ans.classification.validate(model, val_loader)
print(f"after init: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}")
best_val_acc = 0.0
for epoch in range(num_epochs):
    # Trenovani
    train_loss, train_acc = ans.classification.train_epoch(model, train_loader, learning_rate=learning_rate)

    # Validace
    val_loss, val_acc = ans.classification.validate(model, val_loader)
    print(f"Epoch {epoch + 1}: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}")

    # Uloz nejlepsi model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save('../output/two_layer_perceptron_weights.pt')

print(f"Training complete. Best validation accuracy achieved: {best_val_acc:.3f}")
########################################

In [ ]:
test_two_layer_perceptron.TestValAccuracy45.eval(preprocess_fn=preprocess)

In [ ]:
test_two_layer_perceptron.TestValAccuracy50.eval(preprocess_fn=preprocess)